# Observability & Debugging — Part 2: Trace Hierarchy

This notebook teaches you how to read and interpret the span hierarchy that Strands
Agents produces for every invocation.

**What you'll learn:**
- The four span types: Agent, Cycle, Model Invoke, Tool
- How parent-child relationships form the trace tree
- Key attributes on each span type
- How to use `trace_utils.py` for formatted output

**Prerequisites:**
- Complete [01_tracing_setup.ipynb](01_tracing_setup.ipynb) first

In [ ]:
import sys
sys.path.insert(0, ".")

from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands.telemetry.config import StrandsTelemetry
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry import trace

from trace_utils import format_trace_tree, print_trace_summary


class SpanCollector(SpanExporter):
    """Simple in-memory span collector for tutorial use."""
    def __init__(self):
        self._spans = []
    def export(self, spans):
        self._spans.extend(spans)
        return SpanExportResult.SUCCESS
    def get_finished_spans(self):
        return list(self._spans)
    def clear(self):
        self._spans = []
    def shutdown(self):
        self._spans = []


# Configure telemetry
telemetry = StrandsTelemetry()
telemetry.setup_console_exporter()

# Add span collector
span_collector = SpanCollector()
provider = trace.get_tracer_provider()
if hasattr(provider, "add_span_processor"):
    provider.add_span_processor(SimpleSpanProcessor(span_collector))

print("✓ Telemetry configured with console + span collector")

## The Four Span Types

Every agent invocation produces this hierarchy:

| Span Type | Key Attributes | What It Tells You |
|-----------|---------------|-------------------|
| **Agent** | `gen_ai.agent.name`, `gen_ai.request.model` | Overall invocation identity |
| **Cycle** | `event_loop.cycle_id` | Which iteration of the loop |
| **Model Invoke** | `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens` | Token consumption |
| **Tool** | `gen_ai.tool.name`, `tool.status` | Which tool ran, success/failure |

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.

    Args:
        expression: A mathematical expression to evaluate.

    Returns:
        The result as a string.
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
    tools=[calculator],
)

## Single Tool Call Trace

A simple math question produces a minimal trace: 2 cycles (one to call the tool,
one to format the response).

In [ ]:
span_collector.clear()
result = agent("What is 42 * 17?")

spans = span_collector.get_finished_spans()
print(f"Agent response: {result}\n")
print("🌳 Trace Tree:\n")
print(format_trace_tree(spans))
print()
print_trace_summary(spans)

## Multi-Tool Call Trace

When the agent calls multiple tools in one cycle, you'll see multiple tool spans
under the same cycle span.

In [ ]:
span_collector.clear()
result = agent("Calculate 15 * 23, then calculate 100 / 4. Give me both results.")

spans = span_collector.get_finished_spans()
print(f"Agent response: {result}\n")
print("🌳 Trace Tree:\n")
print(format_trace_tree(spans))
print()
print_trace_summary(spans)

## Inspecting Span Attributes

Each span carries attributes with detailed metadata. Let's extract the key
information from each span type.

In [ ]:
spans = span_collector.get_finished_spans()

print("📋 Detailed Span Attributes:\n")
for span in spans:
    attrs = span.attributes or {}
    print(f"── {span.name} ──")
    print(f"   trace_id: {format(span.context.trace_id, '032x')[:16]}...")
    print(f"   status: {span.status.status_code.name}")

    if "gen_ai.usage.input_tokens" in attrs:
        print(f"   input_tokens: {attrs['gen_ai.usage.input_tokens']}")
        print(f"   output_tokens: {attrs.get('gen_ai.usage.output_tokens', 0)}")
    if "gen_ai.tool.name" in attrs:
        print(f"   tool_name: {attrs['gen_ai.tool.name']}")
        print(f"   tool_status: {attrs.get('tool.status', 'N/A')}")
    if "event_loop.cycle_id" in attrs:
        print(f"   cycle_id: {attrs['event_loop.cycle_id']}")
    print()

## Understanding Cycle Count

The number of cycles tells you how many round-trips the agent needed:

- **1 cycle** — agent answered directly without tools
- **2 cycles** — typical: call tool(s), then format response
- **3+ cycles** — complex reasoning, retries, or multi-step tool use
- **5+ cycles** — potential issue (infinite loop, poor tool design)

More cycles = more tokens consumed = higher latency = higher cost.

In [ ]:
spans = span_collector.get_finished_spans()

cycle_count = sum(
    1 for s in spans if "event_loop.cycle_id" in (s.attributes or {})
)
tool_count = sum(
    1 for s in spans if "gen_ai.tool.name" in (s.attributes or {})
)

print(f"📊 Execution Summary:")
print(f"   Cycles: {cycle_count}")
print(f"   Tool calls: {tool_count}")
print(f"   Total spans: {len(spans)}")

if cycle_count <= 2:
    print("   ✓ Efficient execution")
elif cycle_count <= 4:
    print("   ⚠️  Moderate — check if all cycles are necessary")
else:
    print("   🚨 High cycle count — investigate for potential issues")

## Next

Continue to [03_debugging_tools.ipynb](03_debugging_tools.ipynb) to learn how to
debug tool failures and context window pressure using trace data.